# Random Forest — validación temporal y selección de configuración
Lourdes Lema · UDLA · Bloque 4 · v01

**Propósito:** evaluar 16 configuraciones de Random Forest en marzo y junio, seleccionar por MAE promedio con igual peso y comparar con RL y H ya verificadas. No se abre ni evalúa septiembre. No se calcula todavía priorización comercial.

**Colab:** guardar una copia en Drive; ejecutar por bloques. Cargar primero `Evidencias_cortes_v06(1).zip` y después `Evidencias_RL_H_junio_v01.zip`, sin descomprimir. El ajuste realiza 32 entrenamientos y puede tardar varios minutos. Al finalizar, compartir `Evidencias_RF_validacion_v01.zip`.

La verificación previa fue local. Esta copia pública no conserva salidas guardadas; la reproducción debe ejecutarse en Colab con las entradas descritas en la guía. Las propuestas numéricas de hiperparámetros son decisiones de implementación de este bloque, no requisitos docentes ni valores óptimos establecidos por la literatura.

## Configuraciones fijadas antes de ejecutar
| Parámetro | Valores | Propósito |
|---|---|---|
| Número de árboles | 200, 400 | Comparar dos tamaños de ensamble con costo moderado; no prueba convergencia. |
| Profundidad máxima | 8, sin límite explícito | Contrastar restricción de complejidad frente a crecimiento permitido por los demás límites. |
| Mínimo de observaciones por hoja | 1, 10 | Comparar mayor flexibilidad y agrupaciones de mayor tamaño. |
| Variables candidatas por división | 3, 5 | Contrastar selección aleatoria de variables con considerar las cinco disponibles. |

Producto cartesiano: 2 × 2 × 2 × 2 = 16 configuraciones. Parámetros fijos: `criterion='squared_error'`, `bootstrap=True`, `random_state=42`, `min_samples_split=2`, `max_samples=None`, `oob_score=False`, `n_jobs=2`. Cada árbol usa una muestra con reemplazo del tamaño del entrenamiento. No se usa evaluación OOB como sustituto de validación temporal. Se registra el resto de parámetros efectivos.

Semilla 42 fija para reproducibilidad, no seleccionada por desempeño; no se evalúa sensibilidad a semillas. En caso de empate exacto de MAE sin redondear entre configuraciones RF, se elige el identificador RF ascendente fijado en la tabla. No se amplía la búsqueda después de ver resultados sin registrar un nuevo diseño. El mejor resultado es el mejor entre estas opciones, no un óptimo global.

Se conservan cinco predictores, signos, repeticiones, ventanas y clientes elegibles. Sin escalado, imputación, recorte de valores o transformación del objetivo. H no es predictor.


## Carga de datos y referencia de comparación
**Propósito:** usar exactamente los cortes y las métricas verificadas.
**Operaciones:** comprobar huellas, leer entrenamiento inicial y validación, cargar métricas de RL/H. No abrir el archivo de prueba.
**Resultado esperado:** 8.939 filas iniciales, 5.996 de validación y cuatro filas de métricas previas.
**Interpretación:** diferencias de huella detienen el bloque para revisar la versión.


In [ ]:
from pathlib import Path
from datetime import datetime, timezone
import io, json, hashlib, platform, zipfile
import pandas as pd
import numpy as np
import sklearn
from sklearn.ensemble import RandomForestRegressor
from itertools import product
from time import perf_counter
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

RUTA_CORTES = None  # En Colab dejar None.
RUTA_PREVIAS = None   # En Colab dejar None.
def solicitar_zip(ruta_local, mensaje):
    if ruta_local is not None:
        return Path(ruta_local)
    from google.colab import files
    print(mensaje)
    subidos = files.upload()
    nombres = [n for n in subidos if n.lower().endswith('.zip')]
    if len(nombres) != 1:
        raise ValueError('Seleccione un solo ZIP en cada solicitud.')
    return Path(nombres[0])
def leer_miembro(z, nombre):
    encontrados = [n for n in z.namelist() if Path(n).name == nombre]
    assert len(encontrados) == 1, f'Archivo ausente o ambiguo: {nombre}'
    return z.read(encontrados[0])
ruta_cortes = solicitar_zip(RUTA_CORTES, 'Primero: cargue Evidencias_cortes_v06(1).zip')
HUELLAS_CORTES = {'entrenamiento_inicial_v06.csv': '65d7dc09069779841830e0d596ecd4eb6cd78a61f193b12d8a7ffc08a356cd59', 'validacion_v06.csv': 'bfb661e6f35357af531beebfba7f6bf0acffeb126a0a259c129ad4479a8bcedc'}
contenido = {}
with zipfile.ZipFile(ruta_cortes) as z:
    for nombre, huella in HUELLAS_CORTES.items():
        contenido[nombre] = leer_miembro(z, nombre)
        assert hashlib.sha256(contenido[nombre]).hexdigest() == huella, f'Versión distinta: {nombre}'
inicial = pd.read_csv(io.BytesIO(contenido['entrenamiento_inicial_v06.csv']), dtype={'Customer ID':'string'})
validacion = pd.read_csv(io.BytesIO(contenido['validacion_v06.csv']), dtype={'Customer ID':'string'})
ruta_previas = solicitar_zip(RUTA_PREVIAS, 'Segundo: cargue Evidencias_RL_H_junio_v01.zip')
HUELLA_METRICAS_PREVIAS = '394a5fd3df15d3cafe0cf9519cbbe0b24b105118025817e14b7616909364ffe8'
with zipfile.ZipFile(ruta_previas) as z:
    bytes_metricas = leer_miembro(z, 'metricas_por_corte.csv')
    assert hashlib.sha256(bytes_metricas).hexdigest() == HUELLA_METRICAS_PREVIAS, 'Métricas previas distintas de las verificadas.'
    bitacora_previa = json.loads(leer_miembro(z, 'bitacora.json'))
metricas_previas = pd.read_csv(io.BytesIO(bytes_metricas))
assert bitacora_previa['sha256_csv_cortes'] == HUELLAS_CORTES
print('Inicial:',len(inicial),'| Validación disponible:',len(validacion))
print('Métricas previas recuperadas:',len(metricas_previas))
print('scikit-learn:',sklearn.__version__)


## Controles y particiones acumulativas
**Propósito:** comprobar que cada ajuste utiliza únicamente información anterior al corte evaluado.
**Operaciones:** restaurar tipos, validar claves y fórmulas, formar marzo (8.939 → 3.338) y junio (12.277 → 2.658), comprobar fines de etiquetas.
**Resultado esperado:** dos particiones con exactamente cinco columnas predictoras.
**Interpretación:** las observaciones cliente-corte se repiten legítimamente; no se supone independencia estadística ni se aplica partición aleatoria.


In [ ]:
predictores = ['recencia_dias','frecuencia_180d','diversidad_180d',
               'importe_neto_180d','importe_neto_por_compra_180d']
objetivo = 'objetivo_neto_90d'
referencia = 'referencia_neto_90d'
variables = predictores + [referencia, objetivo]
controles = []
def verificar(nombre, condicion):
    controles.append({'control':nombre,'aprobado':bool(condicion)})
    if not condicion: raise AssertionError(nombre)
for nombre,d in [('inicial',inicial),('validacion',validacion)]:
    verificar(nombre+' columnas',set(d.columns)==set(['Customer ID','fecha_corte']+variables))
    d['Customer ID'] = d['Customer ID'].astype('string')
    d['fecha_corte'] = pd.to_datetime(d['fecha_corte'],errors='raise')
    for c in variables: d[c] = pd.to_numeric(d[c],errors='raise')
    verificar(nombre+' sin nulos',not d.isna().any().any())
    verificar(nombre+' claves únicas',not d.duplicated(['Customer ID','fecha_corte']).any())
    verificar(nombre+' finitos',np.isfinite(d[variables]).all().all())
    verificar(nombre+' compras positivas',d.frecuencia_180d.ge(1).all())
    verificar(nombre+' P=M/F',np.allclose(d.importe_neto_por_compra_180d,d.importe_neto_180d/d.frecuencia_180d,rtol=1e-12,atol=1e-9))
cortes_iniciales = pd.to_datetime(['2010-06-01','2010-09-01','2010-12-01'])
verificar('Conteos iniciales',inicial.groupby('fecha_corte').size().to_dict()==dict(zip(cortes_iniciales,[2652,2822,3465])))
verificar('Conteos validacion',validacion.groupby('fecha_corte').size().to_dict()=={
    pd.Timestamp('2011-03-01'):3338,pd.Timestamp('2011-06-01'):2658})
marzo = validacion.loc[validacion.fecha_corte.eq('2011-03-01')].copy()
junio = validacion.loc[validacion.fecha_corte.eq('2011-06-01')].copy()
train = pd.concat([inicial,marzo],ignore_index=True)
for d in [train,junio]:
    d.sort_values(['fecha_corte','Customer ID'],inplace=True)
    d.reset_index(drop=True,inplace=True)
verificar('12277 filas entrenamiento',len(train)==12277)
verificar('2658 filas evaluación',len(junio)==2658)
verificar('Clave acumulada única',not train.duplicated(['Customer ID','fecha_corte']).any())
verificar('Junio fuera del entrenamiento',train.fecha_corte.lt(pd.Timestamp('2011-06-01')).all())
verificar('Etiquetas disponibles',((train.fecha_corte+pd.Timedelta(days=90))<=pd.Timestamp('2011-06-01')).all())
verificar('Métricas previas completas',len(metricas_previas)==4 and not metricas_previas.duplicated(['modelo','corte']).any() and set(metricas_previas.modelo)=={'Referencia H','Regresión lineal'} and set(metricas_previas.corte)=={'2011-03-01','2011-06-01'})
verificar('Métricas previas finitas',np.isfinite(metricas_previas[['MAE_GBP','RMSE_GBP','R2']]).all().all())
particiones = [('2011-03-01',inicial.copy(),marzo.copy()),('2011-06-01',train.copy(),junio.copy())]
for corte,tr,va in particiones:
    tr.sort_values(['fecha_corte','Customer ID'],inplace=True);tr.reset_index(drop=True,inplace=True)
    va.sort_values(['fecha_corte','Customer ID'],inplace=True);va.reset_index(drop=True,inplace=True)
    verificar(corte+' etiquetas disponibles',((tr.fecha_corte+pd.Timedelta(days=90))<=pd.Timestamp(corte)).all())
    print(corte,'entrenamiento:',len(tr),'evaluación:',len(va))


## Registro previo de configuraciones
**Propósito:** dejar documentada la búsqueda antes del primer ajuste.
**Operaciones:** generar el producto cartesiano y guardar un protocolo con fecha, predictores, semilla, criterio y regla de empate.
**Resultado esperado:** 16 identificadores estables RF01–RF16.
**Interpretación:** ninguna configuración se añade o descarta por sus resultados. Se guardan todas, incluidas las que funcionen peor que H.

En la tabla, `NaN` en max_depth representa `None` (sin límite explícito), no un dato faltante de clientes. El JSON conserva el valor null correctamente.


In [ ]:
salida=Path('salidas_RF_validacion_v01');salida.mkdir(exist_ok=True)
configuraciones=[]
for i,(arboles,profundidad,hoja,variables_candidatas) in enumerate(product([200,400],[8,None],[1,10],[3,5]),1):
    configuraciones.append({'id':f'RF{i:02d}','n_estimators':arboles,'max_depth':profundidad,
                           'min_samples_leaf':hoja,'max_features':variables_candidatas})
fijos={'criterion':'squared_error','bootstrap':True,'random_state':42,'min_samples_split':2,
       'max_samples':None,'oob_score':False,'n_jobs':2}
protocolo={'fecha_previa_utc':datetime.now(timezone.utc).isoformat(),'configuraciones':configuraciones,
 'parametros_fijos':fijos,'predictores':predictores,'criterio':'Promedio simple del MAE de marzo y junio',
 'desempate_RF':'MAE sin redondear ascendente, luego identificador RF ascendente',
 'alcance':'No prueba, priorización ni modificación de variables; búsqueda acotada sin garantía de óptimo global.'}
(salida/'protocolo_previo.json').write_text(json.dumps(protocolo,ensure_ascii=False,indent=2))
print(pd.DataFrame(configuraciones).to_string(index=False))


## Ajuste y evaluación de las 16 configuraciones
**Propósito:** comparar distintas complejidades bajo las mismas condiciones temporales.
**Operaciones:** para cada configuración, crear un bosque nuevo en cada corte; ajustar con entrenamiento; predecir y medir MAE, RMSE y R². Guardar predicciones de todas las configuraciones y parámetros efectivos. Verificar métricas mediante fórmulas y que H reproduzca su MAE previo.
**Resultado esperado:** 32 filas de métricas RF y 95.936 predicciones (16 × 5.996).
**Interpretación:** los árboles se ajustan con error cuadrático, mientras la selección usa MAE. Se mantiene esta decisión explícita. Los tiempos son del entorno de ejecución, no una estimación de costo empresarial. No se generan p-valores ni intervalos basados en independencia de las filas.


In [ ]:
filas=[]; predicciones_lista=[]; parametros_efectivos={}
for cfg in configuraciones:
    identificador=cfg['id']; parametros={k:v for k,v in cfg.items() if k!='id'}
    for corte,tr,va in particiones:
        inicio=perf_counter()
        modelo=RandomForestRegressor(**fijos,**parametros)
        modelo.fit(tr[predictores],tr[objetivo])
        pred=modelo.predict(va[predictores]); y=va[objetivo].to_numpy()
        verificar(identificador+' '+corte+' finitos',np.isfinite(pred).all())
        mae=float(mean_absolute_error(y,pred));rmse=float(np.sqrt(mean_squared_error(y,pred)));r2=float(r2_score(y,pred))
        verificar(identificador+' '+corte+' MAE fórmula',np.isclose(mae,np.abs(y-pred).mean(),rtol=1e-12))
        verificar(identificador+' '+corte+' RMSE fórmula',np.isclose(rmse,np.sqrt(((y-pred)**2).mean()),rtol=1e-12))
        h_previo=float(metricas_previas.loc[metricas_previas.modelo.eq('Referencia H') & metricas_previas.corte.eq(corte),'MAE_GBP'].iloc[0])
        verificar(identificador+' '+corte+' H idéntica',np.isclose(mean_absolute_error(y,va[referencia]),h_previo,rtol=1e-12))
        filas.append({'configuracion':identificador,'corte':corte,'n_train':len(tr),'n':len(va),
                      'MAE_GBP':mae,'RMSE_GBP':rmse,'R2':r2,'segundos':perf_counter()-inicio})
        tabla=va[['Customer ID','fecha_corte',objetivo]].copy();tabla['configuracion']=identificador;tabla['prediccion_RF']=pred
        predicciones_lista.append(tabla)
        parametros_efectivos[identificador]=modelo.get_params()
        print(identificador,corte,f'MAE={mae:.6f}',flush=True)
metricas_rf=pd.DataFrame(filas)
predicciones_rf=pd.concat(predicciones_lista,ignore_index=True)
verificar('32 evaluaciones',len(metricas_rf)==32)
verificar('95936 predicciones',len(predicciones_rf)==95936)
verificar('Claves predicciones únicas',not predicciones_rf.duplicated(['configuracion','Customer ID','fecha_corte']).any())


## Selección de configuración y comparación de alternativas
**Propósito:** identificar el RF con menor MAE promedio, conservando el peso de 50 % por corte.
**Operaciones:** construir tabla de los 16 promedios; ordenar sin redondear y desempatar por ID; contrastar RF seleccionado con RL y H verificadas. Si hay empate exacto entre familias se informa, sin elegir mediante la prueba.
**Resultado esperado:** una configuración RF seleccionada dentro de la búsqueda y una tabla comparativa de tres alternativas.
**Interpretación:** las métricas usadas para seleccionar pueden ser optimistas; falta evaluación final. Menor MAE no demuestra mejor priorización ni aumento de ventas causado por contactar clientes. La selección quedará pendiente del cotejo de reproducción en Colab antes de cerrar el protocolo de prueba.


In [ ]:
resumen_rf=metricas_rf.pivot(index='configuracion',columns='corte',values='MAE_GBP').rename(columns={
 '2011-03-01':'MAE_marzo_GBP','2011-06-01':'MAE_junio_GBP'})
resumen_rf['MAE_promedio_GBP']=(resumen_rf.MAE_marzo_GBP+resumen_rf.MAE_junio_GBP)/2
resumen_rf=resumen_rf.reset_index().sort_values(['MAE_promedio_GBP','configuracion']).reset_index(drop=True)
resumen_rf.columns.name=None
mejor_id=resumen_rf.iloc[0].configuracion
mejor_cfg=next(c for c in configuraciones if c['id']==mejor_id)
comparacion=metricas_previas.pivot(index='modelo',columns='corte',values='MAE_GBP').rename(columns={
 '2011-03-01':'MAE_marzo_GBP','2011-06-01':'MAE_junio_GBP'})
comparacion['MAE_promedio_GBP']=(comparacion.MAE_marzo_GBP+comparacion.MAE_junio_GBP)/2
comparacion=comparacion.reset_index();comparacion.columns.name=None
fila=resumen_rf.iloc[0].to_dict();fila.pop('configuracion');fila['modelo']='Random Forest '+mejor_id
comparacion=pd.concat([comparacion,pd.DataFrame([fila])],ignore_index=True)
h=float(comparacion.loc[comparacion.modelo.eq('Referencia H'),'MAE_promedio_GBP'].iloc[0])
comparacion['reduccion_MAE_frente_H_pct']=100*(h-comparacion.MAE_promedio_GBP)/h if h>0 else np.nan
print(resumen_rf.to_string(index=False,float_format=lambda v:f'{v:.6f}'))
print('Configuración RF seleccionada:',mejor_cfg)
print(comparacion.to_string(index=False,float_format=lambda v:f'{v:.6f}'))
predicciones_seleccionadas=predicciones_rf.loc[predicciones_rf.configuracion.eq(mejor_id)].copy()
seleccion={'configuracion_RF':mejor_cfg,'parametros_efectivos':parametros_efectivos[mejor_id],
 'criterio':'MAE promedio con igual peso por corte','estado':'Selección local; cotejo Colab pendiente, prueba no evaluada',
 'alternativas_menor_MAE':comparacion.loc[comparacion.MAE_promedio_GBP.eq(comparacion.MAE_promedio_GBP.min()),'modelo'].tolist()}


## Evidencias y nota provisional de resultados
**Propósito:** conservar resultados, decisiones y una interpretación breve antes de redactar el capítulo integrado.
**Operaciones:** exportar todas las métricas, predicciones, controles y configuración seleccionada; registrar versiones y huellas; generar una nota de trabajo y descargar el ZIP.
**Resultado esperado:** `Evidencias_RF_validacion_v01.zip`.
**Interpretación:** la nota es provisional, no texto final de prueba. No se afirma significación estadística, superioridad general ni beneficio comercial causal.


In [ ]:
tablas={'configuraciones.csv':pd.DataFrame(configuraciones),'metricas_RF_por_corte.csv':metricas_rf,
 'resumen_RF.csv':resumen_rf,'comparacion_validacion.csv':comparacion,
 'predicciones_RF_todas.csv':predicciones_rf,'predicciones_RF_seleccionada.csv':predicciones_seleccionadas,
 'controles.csv':pd.DataFrame(controles),'metricas_RL_H_previas.csv':metricas_previas}
archivos=[salida/'protocolo_previo.json']
for nombre,tabla in tablas.items():
    p=salida/nombre;tabla.to_csv(p,index=False);archivos.append(p)
for nombre,objeto in [('seleccion_RF.json',seleccion),('parametros_efectivos.json',parametros_efectivos)]:
    p=salida/nombre;p.write_text(json.dumps(objeto,ensure_ascii=False,indent=2));archivos.append(p)
nota=('NOTA PROVISIONAL — VALIDACIÓN, NO PRUEBA FINAL\n'
      'Se evaluaron 16 configuraciones de Random Forest en dos cortes temporales con entrenamiento acumulativo. '
      'La configuración seleccionada dentro de esta búsqueda fue '+mejor_id+'. '
      'El MAE promedio de validación fue '+f'{resumen_rf.iloc[0].MAE_promedio_GBP:.6f}'+' GBP, con igual peso por corte. '
      'La comparación con H y regresión lineal se conserva en comparacion_validacion.csv. '
      'Estos resultados requieren cotejo de reproducción y evaluación final; no demuestran mejora de priorización ni ventas adicionales causadas por la atención comercial.')
p=salida/'nota_resultados_provisional.txt';p.write_text(nota);archivos.append(p)
bitacora={'version':'bloque4_v01','fecha_utc':datetime.now(timezone.utc).isoformat(),
 'versiones':{'python':platform.python_version(),'pandas':pd.__version__,'numpy':np.__version__,'scikit-learn':sklearn.__version__},
 'sha256_cortes':hashlib.sha256(ruta_cortes.read_bytes()).hexdigest(),
 'sha256_evidencia_RL_H':hashlib.sha256(ruta_previas.read_bytes()).hexdigest(),
 'huellas_csv_cortes':HUELLAS_CORTES,'huella_metricas_previas':HUELLA_METRICAS_PREVIAS,
 'seleccion':seleccion,'controles_aprobados':len(controles),'predictores':predictores,
 'limites':['Búsqueda acotada no exhaustiva','Una semilla; estabilidad entre semillas no evaluada','Solo dos cortes de validación','Clientes repetidos e historiales solapados','Sin sensibilidad a repeticiones ni prueba final'],
 'sha256_salidas':{p.name:hashlib.sha256(p.read_bytes()).hexdigest() for p in archivos}}
p=salida/'bitacora.json';p.write_text(json.dumps(bitacora,ensure_ascii=False,indent=2));archivos.append(p)
paquete=Path('Evidencias_RF_validacion_v01.zip')
with zipfile.ZipFile(paquete,'w',zipfile.ZIP_DEFLATED) as z:
    for p in archivos:z.write(p,p.name)
print('Evidencias generadas:',paquete.name)
try:
    from google.colab import files
except ImportError:
    print('Verificación local; descarga automática disponible en Colab.')
else:files.download(str(paquete))


## Preguntas para defender las decisiones
- ¿Qué diferencia existe entre el número de árboles y su profundidad?
- ¿Por qué un bosque más complejo no tiene garantizado un menor error futuro?
- ¿Por qué se selecciona por el promedio de dos cortes y no por el mejor resultado aislado?
- ¿Por qué el resultado de validación del RF seleccionado no sustituye a la prueba de septiembre?

## Fuente técnica
scikit-learn developers. (s. f.). *RandomForestRegressor*. scikit-learn. https://scikit-learn.org/stable/modules/generated/sklearn.ensemble.RandomForestRegressor.html

La documentación fundamenta el significado de los parámetros; los valores de la búsqueda son decisiones propias. Consolidar esta entrada con las referencias existentes, sin duplicarlas, y distinguir las obras del mismo autor sin fecha mediante letras cuando corresponda.
